In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
# Project ID: lowercase, hyphens only (Lakebase Autoscaling requirement)
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-game".lower()).strip('-')

In [ ]:
%pip install databricks-sdk --upgrade "psycopg[binary]>=3.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-game".lower()).strip('-')

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec

import sys
sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
project_name = f"projects/{PROJECT_ID}"

try:
    project = w.postgres.get_project(name=project_name)
    print(f"\u267b\ufe0f Found existing Lakebase Autoscaling project: {project.name}")
except Exception:
    print(f"Creating new Lakebase Autoscaling project: {PROJECT_ID}")
    operation = w.postgres.create_project(
        project=Project(
            spec=ProjectSpec(
                display_name=f"Casper's Game ({CATALOG})",
                pg_version="17"
            )
        ),
        project_id=PROJECT_ID
    )
    project = operation.wait()
    add(CATALOG, "postgres_projects", {"name": project.name, "project_id": PROJECT_ID})
    print(f"\u2705 Created project: {project.name}")

# Get primary R/W endpoint
endpoints = list(w.postgres.list_endpoints(parent=f"{project_name}/branches/production"))
if not endpoints:
    raise RuntimeError(f"No endpoints found for {project_name}/branches/production")
ep_name = endpoints[0].name
print(f"\u2705 Using endpoint: {ep_name}")

In [ ]:
import psycopg
from psycopg.errors import DuplicateDatabase

endpoint = w.postgres.get_endpoint(name=ep_name)
lb_host = endpoint.status.hosts.host
cred = w.postgres.generate_database_credential(endpoint=ep_name)
username = w.current_user.me().user_name

conn_string = (
    f"host={lb_host} dbname=postgres "
    f"user={username} password={cred.token} sslmode=require"
)

with psycopg.connect(conn_string) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print(f"\u2705 Connected: {cur.fetchone()[0][:60]}")

        try:
            cur.execute("CREATE DATABASE game")
            print("\u2705 Created database 'game'")
        except DuplicateDatabase:
            print("\u267b\ufe0f Database 'game' already exists, skipping creation")

print(f"\u2705 Lakebase instance ready: {lb_host}")

In [ ]:
game_conn_string = (
    f"host={lb_host} dbname=game "
    f"user={username} password={cred.token} sslmode=require"
)

with psycopg.connect(game_conn_string) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS quest_state (
                player_id TEXT NOT NULL,
                level INT NOT NULL,
                status TEXT DEFAULT 'locked',
                answer TEXT,
                correct BOOLEAN,
                started_at TIMESTAMP,
                completed_at TIMESTAMP,
                hints_used INT DEFAULT 0,
                score INT DEFAULT 0,
                PRIMARY KEY (player_id, level)
            )
        """)
        print("\u2705 Created table quest_state")

        cur.execute("""
            CREATE TABLE IF NOT EXISTS leaderboard (
                player_id TEXT PRIMARY KEY,
                player_name TEXT NOT NULL,
                total_score INT DEFAULT 0,
                levels_completed INT DEFAULT 0,
                total_hints_used INT DEFAULT 0,
                started_at TIMESTAMP,
                finished_at TIMESTAMP
            )
        """)
        print("\u2705 Created table leaderboard")

print(f"\n\u2705 Game Lakebase setup complete")
print(f"   Project: {PROJECT_ID}")
print(f"   Endpoint: {ep_name}")
print(f"   Database: game")
print(f"   Tables: quest_state, leaderboard")